# Demo Preset Generation

This notebook creates the preset inputs used by the web app demo. The goal is not just to create convenient examples, but to make every example defensible: the tabular values must be traceable, the image examples must come from real held-out BreaKHis predictions, and the fusion cases must be clearly documented as synthetic rather than clinical.


## Research Objective

The web demo needs presets so a marker or viewer can run the models without manually typing thirty tabular measurements or searching the image dataset. Presets are useful only if they are transparent. This notebook therefore records where each value came from, why each image was selected, and how each synthetic fusion result is constructed.

Success criteria:

- produce one app-facing JSON manifest at `outputs_v2/reports/demo_presets.json`;
- produce one audit table at `outputs_v2/reports/demo_preset_evidence.csv`;
- cover three tabular examples: benign, borderline, and malignant;
- cover all binary image variants used by the app model: benign/malignant across 40X, 100X, 200X, and 400X;
- build six synthetic fusion cases that demonstrate concordant, discordant, and borderline behaviour without implying real patient-level multimodal matching.


## Methodological Boundary

The tabular Wisconsin dataset and the BreaKHis image dataset are independent. There is no real patient who has both records in this project. For that reason, fusion presets are deliberately described as **synthetic demonstration cases**. They are useful for showing model behaviour and interface behaviour, but they are not clinical multimodal evidence.

This distinction matters for the dissertation: the app can demonstrate an exploratory fusion mechanism, but the written claim must remain that the fusion branch is a constrained methodological experiment under data scarcity.


## Setup And Source Files

This first code cell keeps all filesystem paths and imports in one place. The `find_project_root()` helper makes the notebook runnable both from the repository root and from inside `dissertation_project/notebooks_v2`. The notebook imports the existing `infer_wisconsin()` helper so that the preset probabilities are computed using the same inference code as the web API.


In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebook_Wisconsin" / "brca.csv").is_file():
            return candidate
        nested = candidate / "dissertation_project"
        if (nested / "notebook_Wisconsin" / "brca.csv").is_file():
            return nested
    raise FileNotFoundError("Could not locate dissertation_project from the current working directory.")


PROJECT_ROOT = find_project_root()
REPO_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.inference import infer_wisconsin  # noqa: E402

WISCONSIN_DATA_PATH = PROJECT_ROOT / "notebook_Wisconsin" / "brca.csv"
WISCONSIN_MODEL_PATH = PROJECT_ROOT / "notebook_Wisconsin" / "model.pt"
WISCONSIN_SCALER_PATH = PROJECT_ROOT / "notebook_Wisconsin" / "scaler.joblib"
IMAGE_PREDICTIONS_PATH = PROJECT_ROOT / "outputs_v2" / "reports" / "breakhis_full_test_predictions.csv"
OUTPUT_JSON = PROJECT_ROOT / "outputs_v2" / "reports" / "demo_presets.json"
OUTPUT_CSV = PROJECT_ROOT / "outputs_v2" / "reports" / "demo_preset_evidence.csv"
SOURCE_URL = "https://breascope-ai.vercel.app/"
MAGNIFICATIONS = ["40X", "100X", "200X", "400X"]


## Tabular Preset Source

The three tabular profiles below are the existing quick presets from the deployed BreaScope AI tabular interface:

- `Typical Benign Tumour`
- `Borderline / Suspicious Case`
- `Typical Malignant Tumour`

They are copied into this notebook rather than being left inside the frontend because the app should consume generated research artifacts, not hide research data inside UI code. Each profile keeps all 30 Wisconsin features because the tabular model expects the complete feature vector.


In [ ]:
BREASCOPE_TABULAR_PRESETS = {
    "tabular-typical-benign": {
        "source_label": "Typical Benign Tumour",
        "label_hint": "benign",
        "description": "Typical benign Wisconsin profile from the original BreaScope AI preset set.",
        "values": {
            "x.radius_mean": 12.1,
            "x.texture_mean": 17.9,
            "x.perimeter_mean": 78.1,
            "x.area_mean": 462.8,
            "x.smoothness_mean": 0.092,
            "x.compactness_mean": 0.08,
            "x.concavity_mean": 0.046,
            "x.concave_pts_mean": 0.025,
            "x.symmetry_mean": 0.181,
            "x.fractal_dim_mean": 0.062,
            "x.radius_se": 0.28,
            "x.texture_se": 1.21,
            "x.perimeter_se": 2.05,
            "x.area_se": 20.4,
            "x.smoothness_se": 0.006,
            "x.compactness_se": 0.02,
            "x.concavity_se": 0.025,
            "x.concave_pts_se": 0.008,
            "x.symmetry_se": 0.018,
            "x.fractal_dim_se": 0.0034,
            "x.radius_worst": 13.4,
            "x.texture_worst": 25.0,
            "x.perimeter_worst": 87.0,
            "x.area_worst": 535.5,
            "x.smoothness_worst": 0.12,
            "x.compactness_worst": 0.185,
            "x.concavity_worst": 0.19,
            "x.concave_pts_worst": 0.075,
            "x.symmetry_worst": 0.275,
            "x.fractal_dim_worst": 0.08,
        },
    },
    "tabular-borderline-suspicious": {
        "source_label": "Borderline / Suspicious Case",
        "label_hint": "borderline",
        "description": "Borderline Wisconsin profile from the original BreaScope AI preset set.",
        "values": {
            "x.radius_mean": 14.5,
            "x.texture_mean": 19.5,
            "x.perimeter_mean": 96.0,
            "x.area_mean": 680.0,
            "x.smoothness_mean": 0.1,
            "x.compactness_mean": 0.12,
            "x.concavity_mean": 0.1,
            "x.concave_pts_mean": 0.055,
            "x.symmetry_mean": 0.185,
            "x.fractal_dim_mean": 0.063,
            "x.radius_se": 0.4,
            "x.texture_se": 1.3,
            "x.perimeter_se": 2.9,
            "x.area_se": 42.0,
            "x.smoothness_se": 0.0075,
            "x.compactness_se": 0.027,
            "x.concavity_se": 0.032,
            "x.concave_pts_se": 0.012,
            "x.symmetry_se": 0.021,
            "x.fractal_dim_se": 0.0041,
            "x.radius_worst": 16.7,
            "x.texture_worst": 26.5,
            "x.perimeter_worst": 112.0,
            "x.area_worst": 910.0,
            "x.smoothness_worst": 0.135,
            "x.compactness_worst": 0.27,
            "x.concavity_worst": 0.28,
            "x.concave_pts_worst": 0.12,
            "x.symmetry_worst": 0.3,
            "x.fractal_dim_worst": 0.084,
        },
    },
    "tabular-typical-malignant": {
        "source_label": "Typical Malignant Tumour",
        "label_hint": "malignant",
        "description": "Typical malignant Wisconsin profile from the original BreaScope AI preset set.",
        "values": {
            "x.radius_mean": 17.5,
            "x.texture_mean": 21.0,
            "x.perimeter_mean": 115.0,
            "x.area_mean": 990.0,
            "x.smoothness_mean": 0.105,
            "x.compactness_mean": 0.145,
            "x.concavity_mean": 0.16,
            "x.concave_pts_mean": 0.09,
            "x.symmetry_mean": 0.195,
            "x.fractal_dim_mean": 0.065,
            "x.radius_se": 0.55,
            "x.texture_se": 1.6,
            "x.perimeter_se": 3.8,
            "x.area_se": 55.0,
            "x.smoothness_se": 0.009,
            "x.compactness_se": 0.035,
            "x.concavity_se": 0.045,
            "x.concave_pts_se": 0.017,
            "x.symmetry_se": 0.025,
            "x.fractal_dim_se": 0.0052,
            "x.radius_worst": 20.9,
            "x.texture_worst": 29.5,
            "x.perimeter_worst": 146.0,
            "x.area_worst": 1326.0,
            "x.smoothness_worst": 0.16,
            "x.compactness_worst": 0.35,
            "x.concavity_worst": 0.4,
            "x.concave_pts_worst": 0.18,
            "x.symmetry_worst": 0.33,
            "x.fractal_dim_worst": 0.09,
        },
    },
}


## Validation Helpers

The helper functions make two checks explicit. First, the preset feature names must exactly match the Wisconsin feature order used by the trained model. Second, each value is positioned inside the empirical Wisconsin distribution using a percentile calculation. This gives the evidence CSV enough context to explain whether a preset is low, central, or high relative to the dataset.


In [ ]:
def percentile_position(series: pd.Series, value: float) -> float:
    sorted_values = np.sort(series.astype(float).to_numpy())
    return float(np.searchsorted(sorted_values, value, side="right") / len(sorted_values) * 100)


def ordered_features(raw_values: dict[str, float], feature_order: list[str]) -> dict[str, float]:
    missing = [feature for feature in feature_order if feature not in raw_values]
    extra = [feature for feature in raw_values if feature not in feature_order]
    assert not missing, f"Missing features: {missing}"
    assert not extra, f"Unexpected features: {extra}"
    return {feature: float(raw_values[feature]) for feature in feature_order}


## Building Tabular Cases

This step turns the three copied profiles into app-ready tabular cases. For each preset, the notebook:

- reorders values to match the Wisconsin model input contract;
- asserts that every value is finite;
- checks every feature is inside the observed Wisconsin min/max range;
- runs the published Wisconsin model to record the malignant probability;
- writes one evidence row per feature, including min, max, percentile, and source label.

This is why the final demo can justify the exact numbers instead of saying they were arbitrary examples.


In [ ]:
def build_tabular_cases(wisconsin_df: pd.DataFrame, feature_order: list[str]) -> tuple[list[dict], list[dict]]:
    evidence_rows = []
    tabular_cases = []
    feature_summary = wisconsin_df[feature_order].agg(["min", "max"])

    for preset_id, preset in BREASCOPE_TABULAR_PRESETS.items():
        features = ordered_features(preset["values"], feature_order)
        feature_frame = pd.DataFrame([features], columns=feature_order)
        assert np.isfinite(feature_frame.to_numpy()).all(), f"Non-finite value in {preset_id}"

        probability = float(
            infer_wisconsin(WISCONSIN_MODEL_PATH, WISCONSIN_SCALER_PATH, feature_frame)
            .iloc[0]["probability_malignant"]
        )
        feature_percentiles = []

        for feature, value in features.items():
            dataset_min = float(feature_summary.loc["min", feature])
            dataset_max = float(feature_summary.loc["max", feature])
            percentile = percentile_position(wisconsin_df[feature], value)
            assert dataset_min <= value <= dataset_max, f"{preset_id}:{feature} outside Wisconsin range"
            feature_percentiles.append(percentile)
            evidence_rows.append(
                {
                    "row_type": "tabular_feature",
                    "preset_id": preset_id,
                    "case_family": "tabular",
                    "label_hint": preset["label_hint"],
                    "source_label": preset["source_label"],
                    "feature": feature,
                    "value": value,
                    "dataset_min": dataset_min,
                    "dataset_max": dataset_max,
                    "percentile": percentile,
                    "model_probability_malignant": probability,
                    "source_url": SOURCE_URL,
                    "selection_reason": "Exact profile copied from the deployed BreaScope AI preset set and validated against Wisconsin feature ranges.",
                }
            )

        tabular_cases.append(
            {
                "id": preset_id,
                "labelHint": preset["label_hint"],
                "description": preset["description"],
                "features": features,
                "probabilityMalignant": probability,
                "sourceLabel": preset["source_label"],
                "sourceUrl": SOURCE_URL,
                "evidence": {
                    "percentileMin": float(np.min(feature_percentiles)),
                    "percentileMedian": float(np.median(feature_percentiles)),
                    "percentileMax": float(np.max(feature_percentiles)),
                    "selectionReason": "Preset profile from the existing BreaScope AI demo, kept editable in the new app.",
                },
            }
        )

    return tabular_cases, evidence_rows


## Selecting Image Presets From Held-Out Predictions

The image presets must come from real BreaKHis images. They are selected from `breakhis_full_test_predictions.csv`, which is the corrected patient-level holdout prediction table from the image workflow.

The selection rule is intentionally conservative: within each `label x magnification` group, only correctly predicted holdout images are eligible, and the chosen image is the one nearest to the median malignant probability for that group. This avoids cherry-picking the easiest or most extreme examples while still selecting images the model handles consistently.

The result is exactly eight image presets: benign and malignant examples at 40X, 100X, 200X, and 400X.


In [ ]:
def select_image_cases(predictions_df: pd.DataFrame) -> tuple[list[dict], list[dict]]:
    evidence_rows = []
    image_cases = []
    predictions_df = predictions_df.copy()
    predictions_df["correct"] = predictions_df["y_true"] == predictions_df["y_pred"]

    for label in ["benign", "malignant"]:
        for magnification in MAGNIFICATIONS:
            stratum = predictions_df[
                (predictions_df["label"] == label)
                & (predictions_df["magnification"] == magnification)
                & predictions_df["correct"]
            ].copy()
            assert not stratum.empty, f"No correct holdout images for {label} {magnification}"
            median_probability = float(stratum["y_prob"].median())
            stratum["distance_to_median"] = (stratum["y_prob"] - median_probability).abs()
            selected = stratum.sort_values(["distance_to_median", "filepath"]).iloc[0]
            absolute_path = Path(selected["filepath"])
            assert absolute_path.is_file(), f"Missing selected image: {absolute_path}"
            relative_path = absolute_path.relative_to(PROJECT_ROOT).as_posix()
            image_id = f"{label}-{magnification.lower().replace('x', 'x')}-representative"
            probability = float(selected["y_prob"])
            selection_reason = (
                f"Correct held-out {label} {magnification} prediction nearest to the stratum median "
                f"malignant probability ({median_probability:.6f})."
            )

            image_case = {
                "id": f"image-{image_id}",
                "imageId": image_id,
                "labelHint": label,
                "description": f"Representative {label} BreaKHis tile at {magnification} magnification.",
                "relativePath": relative_path,
                "patientId": str(selected["patient_id"]),
                "magnification": magnification,
                "probabilityMalignant": probability,
                "selectionReason": selection_reason,
            }
            image_cases.append(image_case)
            evidence_rows.append(
                {
                    "row_type": "image_preset",
                    "preset_id": image_case["id"],
                    "case_family": "image",
                    "label_hint": label,
                    "image_id": image_id,
                    "patient_id": selected["patient_id"],
                    "magnification": magnification,
                    "relative_path": relative_path,
                    "model_probability_malignant": probability,
                    "stratum_median_probability": median_probability,
                    "distance_to_median": float(selected["distance_to_median"]),
                    "selection_reason": selection_reason,
                }
            )

    return image_cases, evidence_rows


## Constructing Synthetic Fusion Story Cases

The fusion presets reuse the tabular and image presets built above. They are story cases for the demo rather than new clinical records.

The six cases are designed to show different behaviours:

- concordant benign: benign tabular + benign image;
- concordant malignant: malignant tabular + malignant image;
- discordant stress test: benign tabular + malignant image;
- discordant stress test: malignant tabular + benign image;
- borderline tabular + benign image;
- borderline tabular + malignant image.

The probability is calculated using the same late-fusion rule as the app: average the tabular malignant probability and the image malignant probability. Keeping that rule here makes the manifest traceable to the web result shown in the interface.


In [ ]:
def build_fusion_cases(tabular_cases: list[dict], image_cases: list[dict]) -> tuple[list[dict], list[dict]]:
    tabular_by_id = {case["id"]: case for case in tabular_cases}
    image_by_id = {case["imageId"]: case for case in image_cases}
    benign_image = image_by_id["benign-100x-representative"]
    malignant_image = image_by_id["malignant-100x-representative"]

    specs = [
        ("fusion-concordant-benign", "tabular-typical-benign", benign_image, "benign", "concordant", "Benign tabular preset paired with a benign BreaKHis tile."),
        ("fusion-concordant-malignant", "tabular-typical-malignant", malignant_image, "malignant", "concordant", "Malignant tabular preset paired with a malignant BreaKHis tile."),
        ("fusion-discordant-benign-tabular-malignant-image", "tabular-typical-benign", malignant_image, "benign", "discordant", "Benign tabular preset paired with a malignant tile to stress-test synthetic disagreement."),
        ("fusion-discordant-malignant-tabular-benign-image", "tabular-typical-malignant", benign_image, "malignant", "discordant", "Malignant tabular preset paired with a benign tile to stress-test synthetic disagreement."),
        ("fusion-borderline-with-benign-image", "tabular-borderline-suspicious", benign_image, "borderline", "borderline", "Borderline tabular preset paired with a benign tile."),
        ("fusion-borderline-with-malignant-image", "tabular-borderline-suspicious", malignant_image, "borderline", "borderline", "Borderline tabular preset paired with a malignant tile."),
    ]

    fusion_cases = []
    evidence_rows = []
    for preset_id, tabular_id, image_case, label_hint, case_type, description in specs:
        tabular_case = tabular_by_id[tabular_id]
        tabular_probability = float(tabular_case["probabilityMalignant"])
        image_probability = float(image_case["probabilityMalignant"])
        fusion_probability = float(np.mean([tabular_probability, image_probability]))
        selection_reason = (
            "Synthetic demo case built from independent tabular and image datasets; probability uses the app's current late-fusion mean."
        )
        fusion_cases.append(
            {
                "id": preset_id,
                "imageId": image_case["imageId"],
                "labelHint": label_hint,
                "description": description,
                "features": tabular_case["features"],
                "tabularPresetId": tabular_id,
                "imagePresetId": image_case["id"],
                "caseType": case_type,
                "probabilityMalignant": fusion_probability,
                "tabularProbabilityMalignant": tabular_probability,
                "imageProbabilityMalignant": image_probability,
                "selectionReason": selection_reason,
            }
        )
        evidence_rows.append(
            {
                "row_type": "fusion_preset",
                "preset_id": preset_id,
                "case_family": "fusion",
                "label_hint": label_hint,
                "tabular_preset_id": tabular_id,
                "image_preset_id": image_case["id"],
                "image_id": image_case["imageId"],
                "fusion_case_type": case_type,
                "tabular_probability_malignant": tabular_probability,
                "image_probability_malignant": image_probability,
                "model_probability_malignant": fusion_probability,
                "selection_reason": selection_reason,
            }
        )

    return fusion_cases, evidence_rows


## Manifest Assembly And Assertions

The final generation function combines the tabular, image, and fusion presets into one JSON document. The assertions are part of the research evidence: they fail loudly if the app-facing artifact loses a feature, misses an image variant, uses a missing file, or creates the wrong number of fusion cases.

These checks protect the demo from silent drift. If the upstream predictions change later, rerunning this notebook will either regenerate a valid manifest or stop at the exact broken assumption.


In [ ]:
def generate_demo_presets() -> tuple[dict, pd.DataFrame]:
    wisconsin_df = pd.read_csv(WISCONSIN_DATA_PATH)
    feature_order = [column for column in wisconsin_df.columns if column not in {"Unnamed: 0", "y"}]
    predictions_df = pd.read_csv(IMAGE_PREDICTIONS_PATH)

    tabular_cases, tabular_evidence = build_tabular_cases(wisconsin_df, feature_order)
    image_cases, image_evidence = select_image_cases(predictions_df)
    fusion_cases, fusion_evidence = build_fusion_cases(tabular_cases, image_cases)

    assert len(feature_order) == 30
    assert all(list(case["features"].keys()) == feature_order for case in tabular_cases)
    assert len(tabular_cases) == 3
    assert len(image_cases) == 8
    assert {(case["labelHint"], case["magnification"]) for case in image_cases} == {
        (label, magnification) for label in ["benign", "malignant"] for magnification in MAGNIFICATIONS
    }
    assert all((PROJECT_ROOT / case["relativePath"]).is_file() for case in image_cases)
    assert len(fusion_cases) == 6
    assert {case["caseType"] for case in fusion_cases} == {"concordant", "discordant", "borderline"}

    manifest = {
        "schemaVersion": 1,
        "generatedAt": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": {
            "tabularPresetSourceUrl": SOURCE_URL,
            "wisconsinData": WISCONSIN_DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
            "imagePredictions": IMAGE_PREDICTIONS_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        "disclaimer": "Research demo only. Synthetic fusion cases combine independent datasets and are not real patient-level multimodal records.",
        "featureOrder": feature_order,
        "tabular": tabular_cases,
        "image": image_cases,
        "fusion": fusion_cases,
    }

    evidence = pd.DataFrame(tabular_evidence + image_evidence + fusion_evidence)
    OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_JSON.write_text(json.dumps(manifest, indent=2) + "\n")
    evidence.to_csv(OUTPUT_CSV, index=False)
    return manifest, evidence


## Generate Outputs

This cell writes the two artifacts consumed by the project:

- `demo_presets.json` is read by the FastAPI `/demo-cases` endpoint and by the frontend fallback route;
- `demo_preset_evidence.csv` is the audit trail for the dissertation and demo defence.

The short printed summary is intentionally small so the notebook remains readable when executed top-to-bottom.


In [ ]:
manifest, evidence = generate_demo_presets()
print(f"Wrote {OUTPUT_JSON.relative_to(PROJECT_ROOT)}")
print(f"Wrote {OUTPUT_CSV.relative_to(PROJECT_ROOT)}")
print(f"Tabular: {len(manifest['tabular'])}, image: {len(manifest['image'])}, fusion: {len(manifest['fusion'])}")
evidence.groupby("row_type").size()


## Interpreting The Output Manifest

The JSON manifest is the operational artifact. It contains the feature order, the tabular preset values, the selected image paths, and the synthetic fusion definitions. The web app uses those entries to pre-fill the UI, but the user can still edit tabular fields or upload a different image.

The evidence CSV is the methodological artifact. It explains why the generated values and images were chosen: tabular rows include Wisconsin range and percentile context, image rows include the median-selection rationale, and fusion rows include the exact probability construction.


## Research Notes And Limitations

The preset design is intentionally pragmatic for a live demonstration. It gives broad coverage without overwhelming the page.

Important limitations to state in the dissertation/demo:

- Wisconsin tabular presets are example morphology profiles, not real patient advice.
- BreaKHis image presets are real dataset images, but they are selected for demonstration coverage.
- Fusion presets join independent datasets and therefore are not real multimodal clinical records.
- The fusion probability is a simple late-fusion average, chosen for transparency in the demo rather than as a claim of clinical optimality.

These notes should stay aligned with the app disclaimer and the dissertation framing around exploratory synthetic multimodal work.


## Outputs

- `outputs_v2/reports/demo_presets.json`: app-facing manifest consumed by `/demo-cases` and dataset image routes.
- `outputs_v2/reports/demo_preset_evidence.csv`: audit table with feature ranges, percentiles, image selection evidence, and fusion probability construction.
